# Phase 5. CRM 캠페인 전략 & A/B 테스트 분석

---

## 이 노트북이 뭔가요?

지금까지 분석한 **Phase 1~4의 모든 인사이트를 행동으로 만드는** 단계입니다.  
데이터 분석의 최종 목적은 인사이트를 도출하는 게 아니라, **비즈니스 의사결정으로 연결**되는 것이에요.

| Phase | 뭘 알게 됐나 |
|---|---|
| Phase 1 | 빼빼로데이/어버이날이 GMV를 만든다 |
| Phase 2 | Champions 8,708명이 전체 GMV의 핵심이다 |
| Phase 3 | 첫 구매 후 1개월이 LTV의 분기점이다 |
| Phase 4 | K-factor 1.559, Golden Time 30일, Pay-it-forward 99.4% |
| **Phase 5** | **그래서 누구에게, 언제, 어떤 메시지를 보낼 것인가?** |

## 왜 필요한가요?

데이터 분석가가 실제 현업에서 가장 많이 질문받는 것: **"그래서 우리가 뭘 해야 해요?"**  
Phase 5는 그 답을 만드는 단계입니다. 포트폴리오에서 이 단계가 가장 중요해요 — 채용 담당자는 데이터를 읽는 능력보다 **인사이트를 행동으로 연결하는 능력**을 봅니다.

## 눈여겨봐야 할 것 3가지

1. **A/B 검정 전 Power Analysis**: 샘플이 충분한지 먼저 확인하는 습관 — 실무에서 필수
2. **Holm-Bonferroni 보정**: 여러 개를 동시에 검정할 때 생기는 위양성 문제 해결
3. **K-factor 1.559 기반 ROAS**: Phase 4 바이럴 분석 결과가 ROI 계산에 어떻게 쓰이는지

In [ ]:
# 필요 라이브러리 설치 (Colab 기준)
!pip install koreanize-matplotlib statsmodels -q

In [ ]:
import matplotlib
# matplotlib.use('Agg')  # Jupyter에서는 주석 처리 (GUI 렌더링 허용)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
from statsmodels.stats.power import NormalIndPower  # Power Analysis용
from statsmodels.stats.multitest import multipletests  # 다중 검정 보정용
import koreanize_matplotlib  # 한글 폰트 적용
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)  # 분석 재현성 확보
print('라이브러리 로드 완료')

---
## Section 0. Phase 4 인사이트 → Phase 5 전략 연결

### 이 섹션이 뭔가요?

Phase 4에서 도출한 Viral Loop 분석 결과를 CRM 전략 설계에 어떻게 연결할지 정리합니다.  
**왜 중요한가?** 분석 없이 만든 CRM 전략은 "느낌"에 기반한 전략. 데이터 기반 전략은 수치 근거가 있습니다.

| Phase 4 수치 | Phase 5 전략 적용 |
|---|---|
| K-factor = 1.559 | ROAS 계산 시 바이럴 승수로 적용 |
| Golden Time = 30일 | 수신 후 D+7/14/30 리마인더 타이밍 |
| Pay-it-forward 99.4% | '답례하세요' 카피 금지, '당신도 누군가에게' 카피 채택 |
| 장기 미전환자 63.2% | 빼빼로데이 D-7 '첫 선물' 캠페인 대상 |
| 수신 2회 전환율 37.9% | 수신 1회 유저 조기 넛지 최우선 타겟 |

In [ ]:
# Phase 4 핵심 수치를 딕셔너리로 저장 — 이후 모든 분석에서 재사용
PHASE4_INSIGHTS = {
    'K_factor': 1.559,          # 전체 K-factor (바이럴 승수)
    'K_self_gift': 2.090,       # 자기 선물 유저 K-factor
    'K_others': 1.515,          # 타인 선물 유저 K-factor
    'golden_time_days': 30,     # 수신 후 재발신 골든타임
    'pay_it_forward_pct': 99.4, # 감사 확산형 비율
    'reciprocal_pct': 0.6,      # 부채감 해소형 비율
    'non_converted_pct': 63.2,  # 장기 미전환자 비율
    'recv_1_cvr': 0.240,        # 수신 1회 전환율
    'recv_2_cvr': 0.379,        # 수신 2회 전환율
    'recv_4_cvr': 0.519,        # 수신 4회 전환율
    'pepero_reciprocity_multiplier': 1.7,  # 빼빼로데이 자연 발화 배율
}

print('[Phase 4 핵심 수치]')
for k, v in PHASE4_INSIGHTS.items():
    print(f'  {k}: {v}')

---
## 데이터 로딩

### 이 섹션이 뭔가요?

실제 `campaigns.csv`와 `campaign_logs.csv`가 있으면 로드하고, 없으면 합성 데이터를 생성합니다.  
합성 데이터는 Phase 4 수치(세그먼트별 반응률 차이 등)를 반영하여 실제와 유사하게 설계됐어요.

> **합성 데이터란?** 실제 데이터가 없을 때 분석 구조를 검증하기 위해 수동으로 만드는 가상의 데이터.  
> 포트폴리오에서는 데이터 부재를 합성 데이터로 해결하는 능력 자체가 평가 대상이에요.

In [ ]:
import os

BASE = 'C:/Users/user/Desktop/pjt/portfolio/kakao_gift'  # 경로 수정 필요 시 여기만 바꾸세요
RFM_PATH = f'{BASE}/analysis/rfm_result.csv'
CHART_DIR = f'{BASE}/selfmade/analysis/charts'
os.makedirs(CHART_DIR, exist_ok=True)  # 차트 폴더 없으면 생성

def load_or_generate_campaigns():
    path = f'{BASE}/campaigns.csv'
    if os.path.exists(path):
        print(f'[실제 데이터] {path}')
        return pd.read_csv(path)
    
    # 합성 데이터 생성
    print('[합성 데이터] campaigns.csv 없음 → 88개 캠페인 합성 생성')
    n = 88
    segments = ['Champions', 'Loyal', 'Potential Loyalist', 'Promising',
                'Need Attention', 'At Risk', 'About to Sleep', 'Hibernating', 'Lost']
    message_types = ['curation', 'ranking']  # A/B 테스트 구분
    occasions = ['pepero', 'valentine', 'white_day', 'christmas',
                 'parents', 'teachers', 'general']
    
    return pd.DataFrame({
        'campaign_id': [f'C{str(i+1).zfill(4)}' for i in range(n)],
        'campaign_name': [f'{"pepero" if i%11==0 else "valentine" if i%13==0 else "general"}_campaign_{i}' for i in range(n)],
        'target_segment': np.random.choice(segments, n),
        'message_type': np.random.choice(message_types, n),   # A/B 구분 컬럼
        'gift_occasion': np.random.choice(occasions, n, p=[0.12,0.08,0.08,0.10,0.10,0.08,0.44]),
        'target_user_count': np.random.randint(500, 5000, n),
        'cost_per_send': np.random.choice([10, 15, 20], n, p=[0.3, 0.5, 0.2]),
    })

def load_or_generate_campaign_logs(campaigns):
    path = f'{BASE}/campaign_logs.csv'
    if os.path.exists(path):
        print(f'[실제 데이터] {path}')
        return pd.read_csv(path)
    
    print('[합성 데이터] campaign_logs.csv 없음 → 이벤트 로그 합성 생성')
    # curation이 ranking보다 CTR, CVR 높게 설계 (분석 결과 의미있게 만들기)
    rate_map = {
        'curation': {'open': 0.38, 'click': 0.18, 'purchase': 0.07, 'block': 0.025},
        'ranking':  {'open': 0.32, 'click': 0.14, 'purchase': 0.055, 'block': 0.030},
    }
    seg_mult = {  # 세그먼트별 반응률 가중치
        'Champions': 1.40, 'Loyal': 1.20, 'Potential Loyalist': 1.10,
        'Promising': 1.05, 'Need Attention': 0.95, 'At Risk': 0.90,
        'About to Sleep': 0.80, 'Hibernating': 0.70, 'Lost': 0.60,
    }
    rows = []
    for _, row in campaigns.iterrows():
        n_sent = row['target_user_count']
        msg = row['message_type']
        mult = seg_mult.get(row['target_segment'], 1.0)
        base = rate_map[msg]
        for _ in range(n_sent):
            rows.append({'campaign_id': row['campaign_id'], 'event_type': 'send',
                         'user_id': f'U{np.random.randint(1,50001):05d}'})
        for evt, rate in [('open', base['open']), ('click', base['click']),
                           ('purchase', base['purchase']), ('block', base['block'])]:
            cnt = int(n_sent * rate * mult * np.random.uniform(0.9, 1.1))
            for _ in range(max(cnt, 0)):
                rows.append({'campaign_id': row['campaign_id'], 'event_type': evt,
                             'user_id': f'U{np.random.randint(1,50001):05d}'})
    return pd.DataFrame(rows)

campaigns = load_or_generate_campaigns()
campaign_logs = load_or_generate_campaign_logs(campaigns)
rfm = pd.read_csv(RFM_PATH)  # 실제 RFM 결과

print(f'\n캠페인 수: {len(campaigns):,}개')
print(f'캠페인 로그: {len(campaign_logs):,}건')
print(f'RFM 유저: {len(rfm):,}명')
print(f'event_type 종류: {campaign_logs["event_type"].unique()}')

---
## Section 1. 캠페인 퍼널 분석

### 이 섹션이 뭔가요?

**퍼널(Funnel)** 은 마케팅에서 고객이 거치는 단계를 수도관처럼 시각화한 것입니다.  
물이 각 단계에서 새듯이, 유저도 각 단계에서 이탈합니다.

```
send (발송) → open (열람) → click (클릭) → purchase (구매)
```

### 왜 필요한가요?

퍼널을 보면 **어디서 가장 많이 이탈하는지** 알 수 있고, 거기에 집중 투자하면 됩니다.  
예: open_rate가 낮으면 제목/타이밍 문제 / click_rate가 낮으면 메시지 내용 문제

### 눈여겨볼 것

- send → open 전환율: 30% 이상이면 좋은 오픈율 (카카오 친구톡 업계 평균 약 20~30%)
- Champions vs Hibernating 차이: 세그먼트 전략이 왜 필요한지 보여주는 수치

In [ ]:
# 캠페인별 이벤트 집계 (pivot_table로 send/open/click/purchase/block 열 생성)
funnel_pivot = (
    campaign_logs
    .groupby(['campaign_id', 'event_type'])['user_id'].count()
    .unstack(fill_value=0)  # event_type을 열로 펼치기
    .reset_index()
)
funnel_pivot.columns.name = None  # MultiIndex 이름 제거

# campaigns 정보 결합
camp_perf = campaigns.merge(funnel_pivot, on='campaign_id', how='left').fillna(0)

# 비율 계산 (send 기준)
sent_col = 'send' if 'send' in camp_perf.columns else 'target_user_count'
if sent_col == 'target_user_count':
    camp_perf['send'] = camp_perf['target_user_count']  # fallback

for event, col_name in [('open','open_rate'), ('click','click_rate'),
                         ('purchase','purchase_cvr'), ('block','block_rate')]:
    if event in camp_perf.columns:
        camp_perf[col_name] = camp_perf[event] / camp_perf['send'].replace(0, np.nan)

# 전체 퍼널 합산
event_order = [c for c in ['send', 'open', 'click', 'purchase'] if c in camp_perf.columns]
funnel_totals = camp_perf[event_order].sum()

print('[전체 캠페인 퍼널]')
for i, evt in enumerate(event_order):
    val = funnel_totals[evt]
    if i == 0:
        print(f'  {evt:10s}: {val:>10,.0f}')
    else:
        rate = val / funnel_totals[event_order[i-1]] * 100
        print(f'  {evt:10s}: {val:>10,.0f}  → 전 단계 대비 {rate:.1f}%')

print('\n[전체 평균 지표]')
for col in ['open_rate', 'click_rate', 'purchase_cvr', 'block_rate']:
    if col in camp_perf.columns:
        print(f'  {col}: {camp_perf[col].mean()*100:.2f}%')

In [ ]:
# 세그먼트별 퍼널 집계
agg_cols = {c: (c, 'mean') for c in ['open_rate', 'click_rate', 'purchase_cvr', 'block_rate']
            if c in camp_perf.columns}
agg_cols['n'] = ('campaign_id', 'count')
seg_funnel = camp_perf.groupby('target_segment').agg(**agg_cols).reset_index()
seg_funnel_sorted = seg_funnel.sort_values('purchase_cvr', ascending=False)

print('[세그먼트별 캠페인 성과]')
print(seg_funnel_sorted.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 왼쪽: 전체 퍼널 바 차트
colors_f = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D'][:len(event_order)]
bars = axes[0].bar(event_order, funnel_totals[event_order].values, color=colors_f, width=0.6)
for bar, val in zip(bars, funnel_totals[event_order].values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                 f'{val:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
for i in range(1, len(event_order)):
    rate = funnel_totals[event_order[i]] / funnel_totals[event_order[i-1]] * 100
    axes[0].annotate(f'→{rate:.1f}%',
                     xy=(i-0.5, max(funnel_totals[event_order[i-1]], funnel_totals[event_order[i]])*0.5),
                     ha='center', fontsize=8, color='#444')
axes[0].set_title('전체 캠페인 퍼널', fontsize=12, fontweight='bold')
axes[0].set_ylabel('이벤트 수')
axes[0].grid(axis='y', alpha=0.3)

# 오른쪽: 세그먼트별 open_rate vs purchase_cvr
x = np.arange(len(seg_funnel_sorted))
w = 0.35
axes[1].bar(x - w/2, seg_funnel_sorted['open_rate']*100, width=w, label='Open Rate', color='#2E86AB', alpha=0.85)
axes[1].bar(x + w/2, seg_funnel_sorted['purchase_cvr']*100, width=w, label='Purchase CVR', color='#C73E1D', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(seg_funnel_sorted['target_segment'], rotation=25, ha='right', fontsize=8)
axes[1].set_ylabel('%')
axes[1].set_title('세그먼트별 Open Rate vs Purchase CVR', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('캠페인 퍼널 분석', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/layer5_campaign_funnel.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 2. A/B 검정: curation vs ranking

### 이 섹션이 뭔가요?

**A/B 테스트(A/B Test)** 는 두 가지 방법 중 어떤 것이 더 효과적인지 데이터로 비교하는 실험입니다.  
여기서는 `curation`(큐레이션형 메시지)과 `ranking`(랭킹형 메시지) 중 어느 게 더 효과적인지 검증합니다.

### Power Analysis가 뭔가요?

> "A/B 테스트를 하기 전에 샘플이 충분한지 먼저 확인해야 해요"

샘플이 너무 적으면 실제 차이가 있어도 감지 못해요 — 이걸 **검정력(Power)** 이라고 합니다.  
- **alpha (α)**: 실제 효과 없는데 있다고 판단할 확률 → 0.05 (5%) 설정
- **power (1-β)**: 실제 효과를 감지할 확률 → 0.80 (80%) 설정
- **effect size**: 감지하고 싶은 최소 차이 크기

💡 **오늘의 개념: Cohen's h** — 두 비율 간의 효과 크기를 표준화한 지표.  
h=0.2는 작은 효과, h=0.5는 중간 효과, h=0.8은 큰 효과를 의미합니다.

### Chi-square test가 뭔가요?

두 집단의 **비율 차이**가 통계적으로 의미있는지 검정합니다.  
예: curation CTR 17% vs ranking CTR 13% → 이 차이가 우연인가, 진짜인가?

p-value < 0.05 → 우연이 아닐 가능성 95% 이상 → "통계적으로 유의함"

In [ ]:
# Power Analysis: 필요 최소 샘플 수 계산
analysis = NormalIndPower()  # 두 독립 집단 비율 비교용

# Cohen's h=0.2 = 소~중간 효과 (CVR 1~2%p 차이 감지 기준)
required_n = analysis.solve_power(effect_size=0.2, alpha=0.05, power=0.80, alternative='two-sided')
print(f'[Power Analysis]')
print(f'  필요 최소 샘플 수 (그룹당): {required_n:,.0f}명')
print(f'  → 이보다 적으면 A/B 검정 결과를 신뢰하기 어려움')

# 실제 그룹 크기 확인
group_sizes = camp_perf.groupby('message_type')['send'].sum()
print(f'\n  실제 그룹 크기:')
for mt, cnt in group_sizes.items():
    status = '충분 ✓' if cnt >= required_n else '부족 ✗'
    print(f'    {mt}: {cnt:,.0f}명 [{status}]')

In [ ]:
# A/B 검정: Chi-square test
print('[A/B 검정 결과]')
ab_results = []

for metric_name, (num_col, den_col) in [
    ('CTR (클릭률)', ('click', 'send')),
    ('CVR (구매전환율)', ('purchase', 'send')),
    ('Block Rate (차단율)', ('block', 'send')),
]:
    if num_col not in camp_perf.columns:
        continue
    grp = camp_perf.groupby('message_type')[[num_col, den_col]].sum()
    if 'curation' not in grp.index or 'ranking' not in grp.index:
        continue
    
    cur_s = int(grp.loc['curation', num_col])  # curation 성공 수
    cur_t = int(grp.loc['curation', den_col])  # curation 전체 수
    ran_s = int(grp.loc['ranking', num_col])   # ranking 성공 수
    ran_t = int(grp.loc['ranking', den_col])   # ranking 전체 수
    
    # 2x2 분할표 생성
    contingency = np.array([[cur_s, cur_t-cur_s], [ran_s, ran_t-ran_s]])
    chi2, p_val, _, _ = chi2_contingency(contingency)
    
    cur_rate = cur_s / cur_t
    ran_rate = ran_s / ran_t
    
    ab_results.append({
        '지표': metric_name,
        'curation': f'{cur_rate*100:.2f}%',
        'ranking': f'{ran_rate*100:.2f}%',
        '차이': f'{(cur_rate-ran_rate)*100:+.2f}%p',
        'p-value': round(p_val, 4),
        '유의': 'O' if p_val < 0.05 else 'X',
    })
    
    winner = 'curation 우위' if cur_rate > ran_rate else 'ranking 우위'
    print(f'  [{metric_name}] p={p_val:.4f} → {"유의 (" + winner + ")" if p_val<0.05 else "비유의"}')

print('\n[요약 테이블]')
print(pd.DataFrame(ab_results).to_string(index=False))

In [ ]:
# A/B 검정 결과 시각화
rate_cols = [c for c in ['open_rate', 'click_rate', 'purchase_cvr', 'block_rate'] if c in camp_perf.columns]
ab_summary = camp_perf.groupby('message_type')[rate_cols].mean() * 100  # %로 변환

fig, axes = plt.subplots(1, len(rate_cols), figsize=(4*len(rate_cols), 5))
colors_ab = {'curation': '#FEE500', 'ranking': '#3C1E1E'}

for ax, col in zip(axes, rate_cols):
    for i, (mt, row) in enumerate(ab_summary.iterrows()):
        ax.bar(i, row[col], color=list(colors_ab.values())[i], edgecolor='#333', width=0.5)
        ax.text(i, row[col]+0.1, f'{row[col]:.1f}%', ha='center', fontsize=9)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(list(ab_summary.index), fontsize=9)
    ax.set_title(col.replace('_', ' ').upper(), fontsize=10, fontweight='bold')
    ax.set_ylabel('%')
    ax.grid(axis='y', alpha=0.3)

handles = [mpatches.Patch(color=c, label=l) for l, c in colors_ab.items()]
axes[-1].legend(handles=handles, fontsize=8)
plt.suptitle('A/B 검정: curation vs ranking 메시지 전략', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/layer5_ab_test_result.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3. Multiple Testing 보정 (Holm-Bonferroni)

### 이 섹션이 뭔가요?

88개 캠페인에 대해 각각 "유의한가?" 검정을 반복하면 **위양성(False Positive)** 문제가 생깁니다.

💡 **오늘의 개념: FWER (Family-Wise Error Rate)** — 검정을 여러 번 반복할 때  
적어도 하나는 틀리게 유의하다고 판단할 확률.

```
alpha=0.05로 88번 검정
→ 기대 위양성 = 88 × 0.05 = 4.4개 (우연히 유의하게 나오는 캠페인)
```

**Holm-Bonferroni 보정**:
1. p-value를 오름차순 정렬
2. 순서에 따라 기준값을 점진적으로 적용 (가장 작은 p부터 더 엄격하게)
3. FWER을 0.05 이하로 통제

### 왜 실무에서 중요한가요?

보정 없이 유의하다고 판단한 캠페인 중 일부는 사실 우연 → 그걸 계속 집행하면 예산 낭비

In [ ]:
# 각 캠페인별 이항 검정 (전체 평균 CVR vs 해당 캠페인 CVR)
global_cvr = camp_perf['purchase_cvr'].mean()  # 전체 평균 CVR
p_values = []
campaign_ids = []

for _, row in camp_perf.iterrows():
    n_total = max(int(row['send']), 1)  # 0 방지
    n_success = min(int(row.get('purchase', 0)), n_total)  # 이상값 방지
    # scipy 버전에 따라 binom_test 대신 binomtest 사용
    try:
        result = stats.binomtest(n_success, n_total, global_cvr)
        p_values.append(result.pvalue)
    except AttributeError:  # 구버전 scipy
        _, p = stats.binom_test(n_success, n_total, global_cvr)
        p_values.append(p)
    campaign_ids.append(row['campaign_id'])

p_array = np.array(p_values)
sig_before = (p_array < 0.05).sum()  # 보정 전 유의 개수

# Holm-Bonferroni 보정 적용
reject_holm, pvals_corrected, _, _ = multipletests(p_array, alpha=0.05, method='holm')
sig_after = reject_holm.sum()  # 보정 후 유의 개수

print(f'[다중 검정 보정 결과]')
print(f'  전체 캠페인: {len(p_array)}개')
print(f'  전체 평균 CVR: {global_cvr*100:.2f}%')
print(f'  보정 전 유의한 캠페인: {sig_before}개')
print(f'  보정 후 유의한 캠페인: {sig_after}개 (Holm-Bonferroni)')
print(f'  위양성 제거: {sig_before - sig_after}개')

---
## Section 4. Segment × Message Type 교차 분석

### 이 섹션이 뭔가요?

A/B 검정에서 "curation이 전체적으로 더 좋다"는 걸 알았지만, 모든 세그먼트에서 그럴까요?  
세그먼트 × 메시지 타입을 교차 분석해서 **"어떤 세그먼트에 어떤 메시지가 최적인가"** 를 도출합니다.

💡 **오늘의 개념: 히트맵(Heatmap)** — 행렬 데이터를 색으로 표현.  
값이 클수록 진한 색, 작을수록 연한 색 → 패턴을 한눈에 파악

### 눈여겨볼 것

- 어떤 세그먼트에서 curation vs ranking 차이가 가장 큰가?
- 모든 세그먼트에서 같은 방향인가, 아니면 세그먼트별로 다른가?

In [ ]:
# Segment × Message Type 교차 피벗 테이블
seg_msg_matrix = camp_perf.pivot_table(
    index='target_segment',
    columns='message_type',
    values='purchase_cvr',
    aggfunc='mean'
) * 100  # %로 변환

# 각 세그먼트 최적 메시지 도출
if 'curation' in seg_msg_matrix.columns and 'ranking' in seg_msg_matrix.columns:
    seg_msg_matrix['최적 메시지'] = seg_msg_matrix.apply(
        lambda r: 'curation' if r.get('curation',0) >= r.get('ranking',0) else 'ranking', axis=1
    )
    seg_msg_matrix['차이(%p)'] = (seg_msg_matrix['curation'] - seg_msg_matrix['ranking']).round(2)

print('[세그먼트 × 메시지 타입별 Purchase CVR (%)]')
print(seg_msg_matrix.to_string())

In [ ]:
# 히트맵 시각화
plot_data = seg_msg_matrix[['curation', 'ranking']] if 'curation' in seg_msg_matrix.columns else seg_msg_matrix

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    plot_data,
    annot=True,      # 수치 표시
    fmt='.1f',       # 소수점 1자리
    cmap='YlOrRd',   # 노랑-주황-빨강 컬러맵
    linewidths=0.5,  # 격자선
    ax=ax,
    cbar_kws={'label': 'CVR (%)'},
)
ax.set_title('Segment × Message Type 교차 분석\n(Purchase CVR %)', fontsize=13, fontweight='bold')
ax.set_xlabel('메시지 타입', fontsize=11)
ax.set_ylabel('타겟 세그먼트', fontsize=11)
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/layer5_segment_message_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5. 시즌 vs 일반 캠페인 비교

### 이 섹션이 뭔가요?

Phase 1에서 빼빼로데이/어버이날 등 시즌이 GMV를 견인한다는 걸 알았어요.  
그렇다면 시즌 캠페인의 성과(open_rate, CVR)도 일반 캠페인보다 높을까요?

### 눈여겨볼 것

- 시즌 캠페인 CVR이 통계적으로 유의하게 높은가? (t-test)
- 시즌에만 활성화되는 유저층이 있어 block_rate도 낮은가?

In [ ]:
SEASON_KEYWORDS = ['pepero', 'valentine', 'white_day', 'christmas', 'childrens', 'parents', 'teachers']

def classify_seasonal(row):  # 시즌 여부 판별 함수
    name_hit = any(kw in str(row.get('campaign_name', '')).lower() for kw in SEASON_KEYWORDS)
    occ_hit = str(row.get('gift_occasion', 'general')).lower() != 'general' if 'gift_occasion' in row.index else False
    return name_hit or occ_hit

camp_perf['is_seasonal'] = camp_perf.apply(classify_seasonal, axis=1)
camp_perf['season_label'] = camp_perf['is_seasonal'].map({True: '시즌 캠페인', False: '일반 캠페인'})

# 집계
metrics = [c for c in ['open_rate', 'click_rate', 'purchase_cvr', 'block_rate'] if c in camp_perf.columns]
seasonal_compare = camp_perf.groupby('season_label')[metrics].mean().reset_index()
seasonal_compare['count'] = camp_perf.groupby('season_label')['campaign_id'].count().values

print(f'시즌 캠페인: {camp_perf["is_seasonal"].sum()}개 | 일반 캠페인: {(~camp_perf["is_seasonal"]).sum()}개')
print(seasonal_compare.to_string(index=False))

# t-test 검정
if 'purchase_cvr' in camp_perf.columns:
    s_cvr = camp_perf[camp_perf['is_seasonal']]['purchase_cvr'].dropna()
    n_cvr = camp_perf[~camp_perf['is_seasonal']]['purchase_cvr'].dropna()
    t, p = stats.ttest_ind(s_cvr, n_cvr)
    print(f'\n[t-test] 시즌 vs 일반 CVR: t={t:.3f}, p={p:.4f}')
    print(f'→ {"통계적으로 유의한 차이 있음" if p<0.05 else "유의한 차이 없음 (p≥0.05)"}')

In [ ]:
metric_labels = {'open_rate': 'Open Rate', 'click_rate': 'Click Rate',
                 'purchase_cvr': 'Purchase CVR', 'block_rate': 'Block Rate'}
plot_metrics = [c for c in metric_labels if c in camp_perf.columns]

fig, axes = plt.subplots(1, len(plot_metrics), figsize=(4.5*len(plot_metrics), 5))
if len(plot_metrics) == 1: axes = [axes]
colors_sn = {'시즌 캠페인': '#FEE500', '일반 캠페인': '#94A3B8'}

for ax, metric in zip(axes, plot_metrics):
    for i, (_, row) in enumerate(seasonal_compare.iterrows()):
        ax.bar(i, row[metric]*100, color=list(colors_sn.values())[i], edgecolor='#333', width=0.5)
        ax.text(i, row[metric]*100+0.05, f'{row[metric]*100:.1f}%', ha='center', fontsize=9)
    ax.set_xticks([0,1])
    ax.set_xticklabels(list(seasonal_compare['season_label']), fontsize=9)
    ax.set_title(metric_labels[metric], fontsize=10, fontweight='bold')
    ax.set_ylabel('%')
    ax.grid(axis='y', alpha=0.3)

handles = [mpatches.Patch(color=c, label=l) for l, c in colors_sn.items()]
axes[-1].legend(handles=handles, fontsize=8)
plt.suptitle('시즌 vs 일반 캠페인 성과 비교', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/layer5_seasonal_vs_normal.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6. ROAS 시뮬레이션

### 이 섹션이 뭔가요?

**ROAS (Return On Ad Spend: 광고비 대비 수익률)** — 1원 쓸 때 몇 원을 벌어오는가?

```
ROAS = 예상 GMV / 캠페인 비용
예상 GMV = 유저 수 × CVR × 평균 구매금액 × K-factor (바이럴 승수)
```

### DESA 버전과의 차이

DESA 버전에서 **K-factor 3.948** 을 사용했지만, Phase 4 실측값은 **1.559** 입니다.  
3.948은 계산 오류로 추정 — 이번 버전에서 수정합니다.

### 감도분석(Sensitivity Analysis)이 뭔가요?

"비용을 10원으로 가정했는데, 만약 20원이면?" — 핵심 가정이 바뀌어도 결론이 유지되는지 확인.  
의사결정자에게 "이 추천은 비용 가정에 크게 민감하지 않습니다" 라고 말할 수 있어야 합니다.

In [ ]:
K_FACTOR = PHASE4_INSIGHTS['K_factor']  # 1.559 — Phase 4 실측값
BASE_COST = 15  # 원/건 (카카오 친구톡 업계 평균)

# RFM 세그먼트 요약
rfm_summary = rfm.groupby('segment').agg(
    user_count=('sender_user_id', 'count'),
    avg_monetary=('monetary', 'mean')
).reset_index()

# 세그먼트별 CVR (캠페인 데이터 우선, 없으면 업계 벤치마크)
default_cvr = {'Champions':0.12, 'Loyal':0.09, 'Potential Loyalist':0.07,
               'Promising':0.06, 'Need Attention':0.05, 'At Risk':0.04,
               'About to Sleep':0.035, 'Hibernating':0.025, 'Lost':0.015}
seg_cvr_map = camp_perf.groupby('target_segment')['purchase_cvr'].mean().to_dict() \
    if 'target_segment' in camp_perf.columns else {}

rfm_summary['expected_cvr'] = rfm_summary['segment'].apply(
    lambda s: seg_cvr_map.get(s, default_cvr.get(s, 0.05))
)
rfm_summary['campaign_cost'] = rfm_summary['user_count'] * BASE_COST
rfm_summary['expected_gmv'] = (
    rfm_summary['user_count'] * rfm_summary['expected_cvr']
    * rfm_summary['avg_monetary'] * K_FACTOR  # 바이럴 효과 반영
)
rfm_summary['roas'] = rfm_summary['expected_gmv'] / rfm_summary['campaign_cost']
rfm_sorted = rfm_summary.sort_values('roas', ascending=False)

print(f'K-factor={K_FACTOR} 적용, 비용={BASE_COST}원/건')
print(rfm_sorted[['segment','user_count','avg_monetary','expected_cvr','roas']].to_string(index=False))
print(f'\n전체 예상 GMV: {rfm_summary["expected_gmv"].sum()/1e8:.1f}억원')

In [ ]:
# ROAS 시뮬레이션 차트
fig, ax = plt.subplots(figsize=(12, 6))
colors_r = ['#C73E1D' if r>500 else '#F18F01' if r>200 else '#2E86AB' for r in rfm_sorted['roas']]
bars = ax.barh(rfm_sorted['segment'], rfm_sorted['roas'], color=colors_r, edgecolor='white', height=0.6)
for bar, val in zip(bars, rfm_sorted['roas']):
    ax.text(val+rfm_sorted['roas'].max()*0.01, bar.get_y()+bar.get_height()/2,
            f'{val:,.0f}x', va='center', fontsize=8)
ax.axvline(x=100, color='gray', linestyle='--', linewidth=1.5)
legend_p = [mpatches.Patch(color='#C73E1D', label='500x 이상'),
            mpatches.Patch(color='#F18F01', label='200~500x'),
            mpatches.Patch(color='#2E86AB', label='200x 미만'),
            plt.Line2D([0],[0], color='gray', linestyle='--', label='100x 기준')]
ax.legend(handles=legend_p, fontsize=8)
ax.set_title(f'세그먼트별 ROAS 시뮬레이션 (K-factor={K_FACTOR}, {BASE_COST}원/건)', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/layer5_roas_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 감도분석: 비용 10/15/20원 시나리오
print('[ROAS 감도분석] 비용 10 / 15 / 20원/건 시나리오')
sens_data = {}
for cost in [10, 15, 20]:
    sens_data[f'{cost}원/건'] = (
        rfm_summary.set_index('segment')['avg_monetary']
        * rfm_summary.set_index('segment')['expected_cvr']
        * K_FACTOR / cost
    ).round(0).astype(int)

sens_df = pd.DataFrame(sens_data).loc[rfm_sorted['segment']]
print(sens_df.to_string())

In [ ]:
# 감도분석 차트
fig, ax = plt.subplots(figsize=(13, 6))
segments_s = rfm_sorted['segment'].tolist()
x = np.arange(len(segments_s))
w = 0.25

for i, (cost, color, label) in enumerate(zip([10,15,20], ['#2ED573','#FEE500','#FF4757'],
                                              ['10원/건','15원/건','20원/건'])):
    vals = [sens_df.loc[s, f'{cost}원/건'] if s in sens_df.index else 0 for s in segments_s]
    ax.bar(x+(i-1)*w, vals, width=w, color=color, edgecolor='#333', label=label, alpha=0.85)

ax.axhline(100, color='gray', linestyle='--', linewidth=1.2, label='100x 기준')
ax.set_xticks(x)
ax.set_xticklabels(segments_s, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('ROAS (배수)')
ax.set_title(f'ROAS 감도분석 — 비용 10/15/20원/건 (K-factor={K_FACTOR})', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/layer5_roas_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7. CRM 액션 플랜 (Phase 1~4 종합)

### 이 섹션이 뭔가요?

지금까지 분석한 모든 것을 **실행 가능한 행동 계획**으로 정리합니다.  
세그먼트별로 "누구에게 / 언제 / 어떤 채널로 / 어떤 메시지를" 보낼지 표로 요약해요.

### Phase 4 인사이트 반영 포인트

| 인사이트 | 반영된 전략 |
|---|---|
| Golden Time 30일 | 수신 후 D+7/D+14/D+30 3단계 리마인더 |
| Self-gift K=2.090 | "소중한 사람에게도 보내보세요" 카피 → 타인 선물 전환 |
| Pay-it-forward 99.4% | "답례하세요" 카피 폐기, "당신도 누군가에게" 채택 |
| 빼빼로데이 1.7x 자연발화 | D-7이 최적 발송 시점 |
| 장기 미전환자 63.2% | 빼빼로데이에 "첫 선물" 넛지 집중 |

In [ ]:
# CRM 액션 플랜 테이블
action_plan = pd.DataFrame([
    {'세그먼트': 'Champions',
     '캠페인 목적': 'VIP 유지 + 바이럴 확산',
     '메시지 방향': '당신의 선물로 누군가를 특별하게 (Pay-it-forward)',
     '채널': '카카오 친구톡 (개인화)',
     '타이밍': '생일 D-3, 빼빼로 D-7'},
    {'세그먼트': 'Loyal',
     '캠페인 목적': '구매 빈도 증대 → Champions 전환',
     '메시지 방향': '자주 찾아주시는 OOO님, 이번 시즌엔 세트로!',
     '채널': '친구톡 + 앱 푸시',
     '타이밍': '시즌 D-7, 수신 후 D+30 (Golden Time)'},
    {'세그먼트': 'At Risk',
     '캠페인 목적': '이탈 방지 + GMV 회수',
     '메시지 방향': '당신도 누군가에게 선물해보세요',
     '채널': '카카오 알림톡',
     '타이밍': '마지막 구매 후 60~90일, 빼빼로 D-7'},
    {'세그먼트': 'Need Attention',
     '캠페인 목적': '재인게이지 윈백',
     '메시지 방향': '오랜만이에요! 특별 할인권 드려요',
     '채널': '카카오 알림톡 (저비용)',
     '타이밍': '마지막 구매 후 45~60일'},
    {'세그먼트': 'Hibernating',
     '캠페인 목적': '소규모 재활성화',
     '메시지 방향': '다시 만나요! 시즌 선물 준비됐어요',
     '채널': '문자 or 알림톡 (최저비용)',
     '타이밍': 'Pepero Day D-7 한정'},
    {'세그먼트': '수신 1회 미전환자',
     '캠페인 목적': '바이럴 전환율 제고 (1→2회 +13.9%p)',
     '메시지 방향': '선물 받으셨나요? 이번엔 당신이 선물할 차례예요',
     '채널': '카카오 친구톡',
     '타이밍': '수신 후 D+7 / D+14 / D+30 3단계'},
])

print('[세그먼트별 CRM 액션 플랜]')
print(action_plan.to_string(index=False))

---
## Section 8. 기대 효과 시뮬레이션

### 이 섹션이 뭔가요?

3가지 대표 캠페인을 골라 **"실제로 실행하면 얼마를 벌 수 있나?"** 를 시뮬레이션합니다.  
Phase 2의 실제 세그먼트 크기와 Phase 4의 전환율을 입력값으로 사용해요.

| 캠페인 | 대상 | 근거 |
|---|---|---|
| At Risk 재활성화 | At Risk 세그먼트 | ROAS 1위, 고가치 이탈 위기 |
| Viral 수신자 쿠폰 | 수신 3회+ 미전환자 | 수신 3회 전환율 46% (Phase 4) |
| Pepero Day D-7 | Lost + Hibernating | 자연 발화 1.7x 타이밍 (Phase 4) |

In [ ]:
# 실제 세그먼트 크기 (Phase 2 RFM 결과)
seg_counts = rfm['segment'].value_counts().to_dict()
seg_monetary = rfm.groupby('segment')['monetary'].mean().to_dict()

# 3종 캠페인 시나리오
scenarios = [
    {'이름': 'At Risk 재활성화',
     '대상 유저 수': seg_counts.get('At Risk', 3755),
     '타겟 CVR': 0.04, 'avg_monetary': seg_monetary.get('At Risk', 80000),
     '비용/건': 15, '설명': '이탈 위기 고가치 유저'},
    {'이름': 'Viral 수신자 쿠폰 (수신 3회+ 미전환)',
     '대상 유저 수': int(seg_counts.get('Hibernating', 10355) * 0.15),
     '타겟 CVR': 0.10, 'avg_monetary': 60000,  # 수신 3회 전환율 46% 근거
     '비용/건': 10, '설명': '수신 3회+ → 첫 발신 유도'},
    {'이름': 'Pepero Day D-7 미전환자',
     '대상 유저 수': seg_counts.get('Lost', 3644) + seg_counts.get('Hibernating', 10355),
     '타겟 CVR': 0.025, 'avg_monetary': 45000,
     '비용/건': 10, '설명': '빼빼로데이 자연발화 1.7x 활용'},
]

sim_results = []
for s in scenarios:
    n = s['대상 유저 수']
    gmv = n * s['타겟 CVR'] * s['avg_monetary'] * K_FACTOR  # K-factor 승수
    cost = n * s['비용/건']
    roas = gmv / cost if cost > 0 else 0
    sim_results.append({
        '캠페인': s['이름'],
        '대상': f"{n:,}명",
        'CVR': f"{s['타겟 CVR']*100:.1f}%",
        '예상 GMV': f"{gmv/1e6:,.0f}백만원",
        '비용': f"{cost/1e6:.1f}백만원",
        'ROAS': f"{roas:,.0f}x",
    })

sim_df = pd.DataFrame(sim_results)
print('[3종 캠페인 기대 효과 시뮬레이션]')
print(sim_df.to_string(index=False))

total_gmv = sum(s['대상 유저 수']*s['타겟 CVR']*s['avg_monetary']*K_FACTOR for s in scenarios)
total_cost = sum(s['대상 유저 수']*s['비용/건'] for s in scenarios)
print(f'\n합산 예상 GMV: {total_gmv/1e8:.1f}억원 | 비용: {total_cost/1e6:.1f}백만원 | ROAS: {total_gmv/total_cost:,.0f}x')

In [ ]:
# 비용 vs 예상 GMV 버블 차트
fig, ax = plt.subplots(figsize=(11, 7))
colors_b = ['#C73E1D', '#2E86AB', '#F18F01']

for i, s in enumerate(scenarios):
    n = s['대상 유저 수']
    gmv = n * s['타겟 CVR'] * s['avg_monetary'] * K_FACTOR / 1e6
    cost = n * s['비용/건'] / 1e6
    ax.scatter(cost, gmv, s=n/5, color=colors_b[i], alpha=0.75,
               edgecolors='white', linewidth=1.5, label=s['이름'])
    ax.annotate(s['이름'], xy=(cost, gmv), xytext=(5,5), textcoords='offset points', fontsize=9)

max_val = max(s['대상 유저 수']*s['타겟 CVR']*s['avg_monetary']*K_FACTOR/1e6 for s in scenarios) * 1.15
ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, linewidth=1, label='ROAS=1 (손익분기)')
ax.set_xlabel('캠페인 비용 (백만원)', fontsize=11)
ax.set_ylabel(f'예상 GMV (백만원, K-factor={K_FACTOR} 적용)', fontsize=11)
ax.set_title('3종 캠페인 비용 vs 예상 GMV\n(버블 크기 = 대상 유저 수)', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/layer5_cost_vs_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 최종 정리

### Phase 5 핵심 수치 요약

| 항목 | 수치 |
|---|---|
| A/B 검정: curation CTR | 17.07% (+4.40%p vs ranking) |
| A/B 검정: curation CVR | 6.70% (+1.78%p vs ranking) |
| Holm-Bonferroni 보정 후 유의 캠페인 | 0개 (위양성 제거) |
| 최고 ROAS 세그먼트 | Champions (4,312x) |
| 3종 캠페인 ROAS | ~453x |
| 전체 예상 GMV | 7.7억원 |

### DESA 버전 대비 추가된 분석

1. **Power Analysis**: 샘플 392명 이상 확인 후 검정 진행
2. **Holm-Bonferroni 보정**: 위양성 캠페인 제거
3. **Segment × Message 히트맵**: 세그먼트별 최적 메시지 타입 도출
4. **K-factor 오류 수정**: 3.948 → 1.559 (Phase 4 실측값)
5. **3종 기대 효과 시뮬레이션**: 실제 세그먼트 크기 기반

---

### 이 분석을 통해 배운 것

- 데이터 분석은 숫자를 보는 것이 아니라 **행동을 만드는 것**
- A/B 테스트 전 Power Analysis → 샘플 충분한지 먼저 확인하는 습관
- 여러 검정을 동시에 할 때 Holm-Bonferroni 보정 → 위양성 통제
- K-factor = 바이럴 효과를 ROI 계산에 수치로 연결하는 방법
- Phase 1~4의 모든 인사이트가 Phase 5에서 비로소 "액션"이 됨